In [1]:
from keys import openai_key, replicate_key, groq_key
from working import JudgedDebate, get_client, Debater, Judge, CollabDebate

In [2]:
# n_debaters = 2

# debaters = [
#     Debater(
#         get_client("openai", openai_key),
#         f"debater {id}"
#     ) 
#     for id in range(n_debaters)
# ]

# judge = Judge(
#     get_client("openai", openai_key)
# )

# debate = JudgedDebate(debaters, judge)

In [3]:
# import sys

# sys.path.append("./mats_sae_training")
# from sae_training.sparse_autoencoder import SparseAutoencoder


# def load_sae(
#     layer: int
# ):
#     local_dir = "./jbloom_dictionaries"
#     filename =  f"final_sparse_autoencoder_gpt2-small_blocks.{layer}.hook_resid_pre_24576.pt"

#     save_path = f"{local_dir}/{filename}"
#     sae = SparseAutoencoder.load_from_pretrained(save_path)
#     sae.to("cuda:0")

#     return sae

In [4]:
from sam_dictionaries.sae import AutoEncoder
import torch

def load_sae(
    layer: int
):
    path = f"./sam_dictionaries/resid_out_layer{layer}/10_32768/ae.pt"

    activation_dim = 512 # dimension of the NN's activations to be autoencoded
    dictionary_size = 64 * activation_dim # number of features in the dictionary
    ae = AutoEncoder(activation_dim, dictionary_size)
    ae.load_state_dict(torch.load(path))
    ae.to("cuda:0")

    return ae

In [5]:
from working import ActivationCache
from nnsight import LanguageModel
from working.config import CacheConfig

layer = 1

# TODO
# 4_4365
# 3_11662
# 2_7440
# 1_7703
# 0_8313

model = LanguageModel("EleutherAI/pythia-70m", device_map="auto", dispatch=True)
sae = load_sae(
    layer=layer
)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [6]:
cfg = CacheConfig

cache = ActivationCache(
    layer=layer,
    model=model,
    sae=sae,
    cfg = cfg
)


100%|██████████| 9/9 [00:31<00:00,  3.51s/it]

Activation Cache Size: torch.Size([1800, 128, 32768])


In [7]:
feature_id = 7703 
examples_list, max_act = cache.get_top_examples(feature_id)

n_debaters = 3

debaters = [
    Debater(
        get_client("openai", openai_key),
        f"debater {id}"
    ) 
    for id in range(n_debaters)
]

debate = CollabDebate(debaters, examples_list)

debate.run(max_rounds=2)

debate.history.save("openai", f"{layer}_{feature_id}")